# 19 · One model told the QP

In notebook 18, one of three models trained on notebook 17's pairs won everywhere: notebook 8's body, told each frame's
QP. It beat the blind model in every QP bin, and a specialist for QP 38 and up only tied it there. Before anything is
split into a low and a high QP model, this notebook takes the one told model further:

- **A second seed.** Notebook 18 trained every model once. Here the told and the blind model are each trained at two
  seeds, so the gap between them can be set against the gap between two seeds of one model.
- **Against what ships.** Notebook 8's graph baked at 0.46 bpp, the one notebook 17 measured, scored on the same pairs.
- **A wrong QP.** The told model given the QP off by a few steps: how exact the number the host sends has to be.
- **Biases that are a line in the QP.** Each shift is a 1x1 convolution of the QP with no bias, so a graph baked for
  one QP has the biases `b + w * QP / 51`. A client could compute them for every frame instead of picking the nearest
  of a few baked graphs. Checked here, not built.

**Limits.** As in notebook 18, the pairs are video only (YouTube targets), so nothing here is about screens yet. The
epoch kept is the best on the same validation pairs that are scored, alike for every model.

Needs: notebook 17, part 5 (`qp_pairs/`), and notebook 8's graph in `dist/upscale/`. Seed 0 of each model is notebook
18's training, loaded from `checkpoints/qp_runs/`; seed 1 is trained into the same folder. Writes
`checkpoints/upscale_qp.pt`.

In [ ]:
"""Settings."""

import os

import numpy as np
import onnx
import torch
import torch.nn as nn
from onnx import numpy_helper
from torch.utils.data import DataLoader, Dataset, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, sources, training, webexport
from helpers.training import DEVICE

PAIRS_DIR = sources.ROOT / "upscale" / "qp_pairs"                  # notebook 17, part 5
RUN_DIR = training.CHECKPOINT_DIR / "qp_runs"                      # notebook 18's trainings, and seed 1's
CHECKPOINT = training.CHECKPOINT_DIR / "upscale_qp.pt"             # the told model at its better seed
SHIPPED = webexport.MODELS_DIR / "upscale_screens_24x2_bpp0.46_tile320x180.onnx"   # notebook 8's, as notebook 17 ran it

QP_EDGES = (0, 18, 26, 30, 34, 38, 42, 46)   # notebook 17's bins of a frame's QP
QP_MAX = 51                                  # the model is told QP / QP_MAX
WIDTH, BLOCKS, STEM = 24, 2, 3               # notebook 8's body
BATCH_SIZE, LEARNING_RATE, MAX_EPOCHS = 32, 1e-3, 500
SEEDS = (0, 1)
QP_ERRORS = (-12, -6, -3, 3, 6, 12)          # how far off the told model is told, in QP steps

training.seed_everything(SEEDS[0])

## 1. Data

Notebook 18's: every patch with its frame's QP, divided by 51 so it runs from 0 to 1. Every loader hands it out; the
blind models drop it.

In [ ]:
manifest = training.load_manifest(PAIRS_DIR)
training.describe(manifest)
SCALE = manifest["scale"]
trim = training.trimmer(manifest)


class WithQP(Dataset):
    """A split's pairs as ((LR, QP / QP_MAX), HR)."""

    def __init__(self, split, augment):
        self.pairs = patches.PatchPairs(PAIRS_DIR, split, manifest, augment=augment)
        self.qp = np.load(PAIRS_DIR / manifest["splits"][split]["qp"])
        self.condition = torch.from_numpy(self.qp / QP_MAX).float()[:, None, None, None]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        lr, hr = self.pairs[index]
        return (lr, self.condition[index]), hr


def batches(dataset, low=0, high=99, train=False):
    """A loader of `dataset`'s patches at QP from `low` up to `high`."""
    kept = Subset(dataset, np.flatnonzero((dataset.qp >= low) & (dataset.qp < high)))
    return DataLoader(kept, batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train)


train_set, val_set = WithQP("train", augment=True), WithQP("val", augment=False)
BINS = list(zip(QP_EDGES[:-1], QP_EDGES[1:]))
ROWS = [("all", 0, 99), *[(f"{low}-{high}", low, high) for low, high in BINS]]   # what every table is split by
print(f"{'patches':<8}{'all':>8}" + "".join(f"{f'{low}-{high}':>8}" for low, high in BINS))
for name, dataset in (("train", train_set), ("val", val_set)):
    print(f"{name:<8}{len(dataset):8}" + "".join(f"{((dataset.qp >= low) & (dataset.qp < high)).sum():8}" for low, high in BINS))

## 2. Models

The told and the blind model are notebook 18's. Notebook 8's graph is read back from its ONNX file into a
`models.LumaUpscaler`: its weights carry the same names, and the model read back is checked against the graph.

**The biases are a line in the QP.** `ToldLuma.baked(c)` adds `shift(c)` to three biases, and each shift is a 1x1
convolution of one number with no bias of its own: `w * c`. So a graph baked for a QP has the biases
`b + w * QP / 51`, with `w` 24 numbers for each of the three. Checked on random weights: a graph baked at any QP has
the biases of the line through QP 0 and QP 51, and computes what the told model computes.

In [ ]:
class Blind(nn.Module):
    """A model that is not told, on loaders that tell: it drops the QP."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, condition=None):
        return self.model(x)


class WrongQP(nn.Module):
    """A told model given every frame's QP off by `error` steps, kept inside 0 to QP_MAX."""

    def __init__(self, model, error):
        super().__init__()
        self.model, self.error = model, error

    def forward(self, x, condition):
        return self.model(x, (condition + self.error / QP_MAX).clamp(0, 1))


def build(told):
    if told:
        return models.ToldLuma(conditions=1, width=WIDTH, blocks=BLOCKS, stem=STEM, scale=SCALE)
    return Blind(models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE))


def from_graph(path):
    """An exported `LumaUpscaler` graph as the torch model again: its initializers are the state dict."""
    weights = {t.name: torch.from_numpy(numpy_helper.to_array(t).copy()) for t in onnx.load(str(path)).graph.initializer}
    model = models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE)
    model.load_state_dict(weights)
    return model.to(DEVICE).eval()


def biases(model, qp):
    """The three biases a graph baked from told `model` for `qp` has, end to end."""
    plain = model.baked(qp / QP_MAX)
    return torch.cat([plain.body[0].bias, *[block.conv.bias for block in plain.body[2:]]])


assert build(told=True).halo <= manifest["halo"], "the model reaches further than the patches' halo"

shipped = from_graph(SHIPPED)
tile = webexport.read_metadata(onnx.load(str(SHIPPED)))
step_h, step_w, halo = int(tile["step_h"]), int(tile["step"]), int(tile["halo"])
probe = np.random.rand(1, 3, step_h + 2 * halo, step_w + 2 * halo).astype(np.float32)
print(f"notebook 8's graph read back: {webexport.check_graph(SHIPPED, shipped, probe):.1e} from the graph itself")

told_probe = build(told=True).to(DEVICE)
with torch.no_grad(), training.full_precision():
    for weight in told_probe.parameters():
        if weight.requires_grad:
            weight.normal_(0, 0.1)
    at_0, at_max = biases(told_probe, 0), biases(told_probe, QP_MAX)
    x = torch.rand(2, 3, 40, 40, device=DEVICE)
    for qp in (7, 23.5, 41):
        off_line = (biases(told_probe, qp) - (at_0 + (at_max - at_0) * qp / QP_MAX)).abs().max().item()
        told_out = told_probe(x, torch.full((2, 1, 1, 1), qp / QP_MAX, device=DEVICE))
        computes = (told_out - told_probe.baked(qp / QP_MAX)(x)).abs().max().item()
        assert off_line < 1e-5, f"baked at QP {qp}, the biases are off the line by {off_line:.1e}"
        assert computes < 1e-5, f"baked at QP {qp}, the graph computes something else ({computes:.1e})"
print(f"a graph baked for a QP: biases b + w * QP / {QP_MAX}, {at_0.numel()} of them, and the told model's output")

## 3. Train

Notebook 18's recipe: Adam from 1e-3, batch 32, `metrics.ssim_l1_loss`, the epoch with the best SSIM-Y kept, until it
has converged. The seed sets the starting weights and then, again, the batches. Seed 0 of each is notebook 18's training
and just loads; seed 1 trains here, about 40 minutes a model on an RTX 3090. Run the cell again after an interruption
and it carries on.

In [ ]:
def run_name(kind, seed):
    """Seed 0 is notebook 18's training, under its name there."""
    return kind if seed == 0 else f"{kind}_seed{seed}"


def train(kind, told, seed):
    """A fresh model trained on every QP, or its saved training loaded. Returns it and its best epoch."""
    torch.manual_seed(seed)
    model = build(told).to(DEVICE)
    torch.manual_seed(seed)
    name = run_name(kind, seed)
    history, best = training.train(model, batches(train_set, train=True), batches(val_set), trim, MAX_EPOCHS,
                                   learning_rate=LEARNING_RATE, loss_fn=metrics.ssim_l1_loss, select="ssim",
                                   checkpoint=RUN_DIR / f"{name}.pt", verbose=0)
    print(f"{name:<16} best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f}")
    return model.eval(), best


trained, bests = {}, {}
for kind, told in (("blind", False), ("told_qp", True)):
    for seed in SEEDS:
        trained[kind, seed], bests[kind, seed] = train(kind, told, seed)

## 4. Gain over bicubic, by QP

Each cell is SSIM-Y and luma dB over bicubic on that bin's validation patches. `notebook 8` is the graph the client runs
today, trained on notebook 8's set (screens, photos and video at known bitrates) and baked for 0.46 bpp: what the
client gets now, not a fair fight between recipes.

The second table is the rulers: the told model's lead over the blind one (the mean of two seeds each), and how far
apart each model's two seeds land.

In [ ]:
bicubic = Blind(models.Resize(SCALE))
columns = {"notebook 8": Blind(shipped), **{f"{kind} s{seed}": model for (kind, seed), model in trained.items()}}

scores, counts = {}, {}   # (row, column) -> training.evaluate's scores; row -> patches
for label, low, high in ROWS:
    loader = batches(val_set, low, high)
    counts[label] = len(loader.dataset)
    scores[label, "bicubic"] = training.evaluate(bicubic, loader, trim)
    for name, model in columns.items():
        scores[label, name] = training.evaluate(model, loader, trim)


def gain(label, name, key):
    return scores[label, name][key] - scores[label, "bicubic"][key]


def mean_gain(label, kind, key):
    return np.mean([gain(label, f"{kind} s{seed}", key) for seed in SEEDS])


def seed_gap(label, kind, key):
    return abs(gain(label, f"{kind} s{SEEDS[0]}", key) - gain(label, f"{kind} s{SEEDS[1]}", key))


print(f"{'QP':>7}{'patches':>9}{'bicubic':>9}" + "".join(f"{name:>18}" for name in columns))
print(f"{'':>25}" + "".join(f"{'ssim-y / dB':>18}" for _ in columns))
for label, _, _ in ROWS:
    cells = [f"{gain(label, name, 'ssim'):+.4f} / {gain(label, name, 'y'):+.2f}" for name in columns]
    print(f"{label:>7}{counts[label]:9}{scores[label, 'bicubic']['ssim']:9.4f}" + "".join(f"{cell:>18}" for cell in cells))

print()
print(f"{'QP':>7}{'told - blind':>20}{'blind s0 - s1':>20}{'told s0 - s1':>20}")
for label, _, _ in ROWS:
    lead = f"{mean_gain(label, 'told_qp', 'ssim') - mean_gain(label, 'blind', 'ssim'):+.4f} / " \
           f"{mean_gain(label, 'told_qp', 'y') - mean_gain(label, 'blind', 'y'):+.2f}"
    rulers = [f"{seed_gap(label, kind, 'ssim'):.4f} / {seed_gap(label, kind, 'y'):.2f}" for kind in ("blind", "told_qp")]
    print(f"{label:>7}{lead:>20}" + "".join(f"{ruler:>20}" for ruler in rulers))

## 5. A wrong QP

The told model at its better seed, given every frame's QP off by each of `QP_ERRORS` (kept inside 0 to 51): SSIM-Y and
luma dB against the same model told the true QP. A frame's QP is what x264 reports for it, so the host knows it
exactly; a client that rounds it, or bakes a few graphs and picks the nearest, is off by a few. Every +6 doubles the
rounding step.

In [ ]:
best_seed = max(SEEDS, key=lambda seed: bests["told_qp", seed]["score"])
told_model = trained["told_qp", best_seed]
print(f"told_qp at seed {best_seed}; error in QP steps")
print(f"{'QP':>7}" + "".join(f"{error:>+16d}" for error in QP_ERRORS))
for label, low, high in ROWS:
    loader = batches(val_set, low, high)
    true = scores[label, f"told_qp s{best_seed}"]
    cells = []
    for error in QP_ERRORS:
        wrong = training.evaluate(WrongQP(told_model, error), loader, trim)
        cells.append(f"{wrong['ssim'] - true['ssim']:+.4f} / {wrong['y'] - true['y']:+.2f}")
    print(f"{label:>7}" + "".join(f"{cell:>16}" for cell in cells))

## 6. Save

The told model at its better seed, with what a later notebook needs to rebuild it and its scores by QP. Rebuilt from the
file alone and baked for a QP, it must compute what the trained model does. Then how far the QP moves the biases, against
their size: how much the model leans on what it is told.

In [ ]:
table = {label: {name: {"ssim": scores[label, name]["ssim"], "y": scores[label, name]["y"]}
                 for name in ["bicubic", *columns]} for label, _, _ in ROWS}
training.save_checkpoint(CHECKPOINT, told_model, architecture="ToldLuma",
                         arguments={"conditions": 1, "width": WIDTH, "blocks": BLOCKS, "stem": STEM},
                         told=["qp"], qp_max=QP_MAX, scale=SCALE, halo=told_model.halo, loss="ssim_l1_loss",
                         select="ssim", seed=best_seed, best_epoch=bests["told_qp", best_seed]["epoch"],
                         val_by_qp=table, qp_edges=list(QP_EDGES))

checkpoint = training.load_checkpoint(CHECKPOINT)
loaded = models.ToldLuma(scale=checkpoint["scale"], **checkpoint["arguments"]).to(DEVICE)
loaded.load_state_dict(checkpoint["state_dict"])
for qp in (20, 40):
    training.assert_same_model(told_model.baked(qp / QP_MAX), loaded.baked(qp / QP_MAX))
print("reloaded model is identical")

with torch.no_grad():
    at_0, at_max = biases(told_model, 0), biases(told_model, QP_MAX)
for layer, (start, end) in zip(("stem", "block 1", "block 2"), ((0, WIDTH), (WIDTH, 2 * WIDTH), (2 * WIDTH, 3 * WIDTH))):
    moved, size = (at_max - at_0)[start:end].abs(), at_0[start:end].abs()
    print(f"{layer:<8} QP 0 to {QP_MAX} moves a bias {moved.mean():.3f} on average (largest {moved.max():.3f}); "
          f"the biases at QP 0 average {size.mean():.3f}")

## 7. What it says

**The told model's lead is not luck.** Seed 1 lands where seed 0 did (0.9124 against 0.9121 SSIM-Y), and in every bin
the two told seeds are at most 0.0004 apart. Its lead over the blind model, the mean of two seeds each, is 0.0026 to
0.0085 SSIM-Y a bin, more than 4 times the larger seed gap in that bin. The lead is largest at low QP: +1.84 dB at
QP 0-18 and +0.77 at 18-26, where a blind model has to hedge between sharpening a clean frame and deblocking a coded
one. The blind seeds also disagree most there (0.21 dB at 0-18).

**It beats what ships in every bin.** On these pairs notebook 8's graph gains +0.0128 SSIM-Y / +0.68 dB over bicubic
and the told model +0.0313 / +1.48. From QP 38 up notebook 8's graph is below bicubic in dB (-0.05 and -0.11), while
the told model is at +0.87 and +0.76. At QP 0-18 it is +0.0154 / +2.26 against +0.0122 / +1.33. The blind model
trained on these pairs is *behind* notebook 8's graph at low QP (+0.0067 against +0.0122 SSIM-Y at 0-18): trained on
every QP and not told which, it gives up the clean end. The QP is what wins it back.

**The QP has to be exact.** Told a QP 3 steps off, the model loses 0.0006 to 0.0026 SSIM-Y in the bins from 18 to
38, where its two seeds are 0.0001 to 0.0002 apart. Six steps off, it is no better than the blind model from QP 26 to 42. A QP that is too high
costs more than one too low by the same amount, because the model smooths more than the frame needs. A few graphs
baked at fixed QPs, with the nearest picked, would give back much of the lead unless they sit about 2 steps apart
(some 20 graphs). The client should instead compute the biases `b + w * QP / 51` for each frame from the QP x264
reported for it: 72 multiply-adds a frame, with the same graph and the same cost.

**The model leans on the QP.** From QP 0 to 51, block 1's biases move 0.111 on average, against 0.042 for the biases
themselves.

Saved as `checkpoints/upscale_qp.pt` (seed 1). Still to do:

- **Low and high QP models, each told the QP**, against this one. To pay, each has to beat it in its own range by
  clearly more than the told seed gap (0.0004 here), say 0.001 SSIM-Y.
- **Screens.** These pairs are video only, and they only use P frames.
- **A graph whose biases the client sets for each frame**, built and timed in the browser, before this model can ship.